# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlaaSherif-Ibrahim/FLYRANK_AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Filled in for **Lane 2 — Refresh / Content Opportunity Scoring**. The contract below is written
against the **warehouse release** (`fact_content_daily_performance`, partition `month=2026-03`),
not the starter CSV — this is the slice of the river my lane actually drinks from.

All development happens on a mid-panel month. The `_sample` table is June 2026, the panel's last
month — the natural outcome window of any past→future label — so it stays sealed this week.

In [1]:
%pip install -q duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\alaa\FLYRANK_AI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
# Setup: token (never printed), DuckDB over hf://, and the one partition this notebook touches.
import os, getpass, platform, tempfile, urllib.request

import duckdb

IN_COLAB = "google.colab" in os.environ or "google.colab" in __import__("sys").modules

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and os.path.exists(".env"):          # local runs: repo-root .env
    for line in open(".env"):
        if line.strip().startswith("HF_TOKEN"):
            HF_TOKEN = line.split("=", 1)[1].strip()
if not HF_TOKEN:
    try:
        from google.colab import userdata             # Colab Secret named HF_TOKEN
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
try:
    con.execute("LOAD httpfs")
except Exception:                                     # some networks redirect the extension CDN;
    mach = {"x86_64": "amd64", "amd64": "amd64",       # fall back to a manual download + local install
            "aarch64": "arm64", "arm64": "arm64"}.get(platform.machine().lower(), platform.machine().lower())
    plat = f"{platform.system().lower()}_{mach}"
    url = f"https://extensions.duckdb.org/v{duckdb.__version__}/{plat}/httpfs.duckdb_extension.gz"
    gz = os.path.join(tempfile.gettempdir(), "httpfs.duckdb_extension.gz")
    urllib.request.urlretrieve(url, gz)
    con.execute(f"INSTALL '{gz}'")
    con.execute("LOAD httpfs")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
del HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

n = con.sql(f"SELECT COUNT(*) FROM {FACT_MAR}").fetchone()[0]
print(f"connected. month=2026-03 partition: {n:,} page-day rows")

connected. month=2026-03 partition: 9,841,378 page-day rows


## 1. Unit of analysis + time window

**The contract, in plain words (five answers):**

1. **One row = one page-day.** In `fact_content_daily_performance`, one row is one content item
   (`content_hash_id`) on one calendar day (`report_date`) for one client. My lane's *working
   frame* then collapses those days into **one row per page**, summing/averaging the feature
   window — so every model decision is about a page, not a day.
2. **Table(s):** `fact_content_daily_performance` only, read from the single partition
   `month=2026-03` (~9.8M rows). `dim_clients` is touched once, in section 4, to measure history
   coverage — never joined into features.
3. **Time window:** decision moment = end of **2026-03-20**. Features come from
   **Mar 1–20** (20 days, strictly before the decision). The label comes from
   **Mar 21–31** (11 days, strictly after). June 2026 (`_sample`) is the sealed test month.
4. **What I predict:** a forward-looking decline flag — did the page's daily impression rate drop
   by ≥ 20% in Mar 21–31 vs its Mar 11–20 baseline? It is an **observed future outcome**, an
   upgrade over W02's same-window rule; still a **proxy** for "this page deserves refresh
   review first", not proof a refresh would help.
5. **Deliberately excluded:** the whole `fact_content_query_90d` table. Its fixed 90-day window
   overlaps my label period near the snapshot edge, so its impression columns can contain my
   outcome window — using them would leak. It also repeats per-content context on every query
   row (an `ANY_VALUE()` trap I don't need this week).


In [3]:
# Access + scale sanity check for the table named in the contract.
n_rows = con.sql(f"SELECT COUNT(*) FROM {FACT_MAR}").fetchone()[0]
n_days = con.sql(f"SELECT COUNT(DISTINCT report_date) FROM {FACT_MAR}").fetchone()[0]
print(f"rows: {n_rows:,} across {n_days} distinct report_dates (expect 31 for March)")

rows: 9,841,378 across 31 distinct report_dates (expect 31 for March)


## 2. Fields: feature / label / context / excluded

Every field this notebook touches, in exactly one bucket:

| Bucket | Fields | Why |
|---|---|---|
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | grouping, joining, client-grouped splits — pseudonymous IDs are never model features |
| **Feature (1)** | `imp_f20` = Σ impressions, Mar 1–20 | visibility scale before the decision moment |
| **Feature (2)** | `ctr_f20` = clicks ÷ impressions, Mar 1–20 | snippet appeal before the decision moment |
| **Feature (3)** | `pos_f20` = mean `gsc_avg_position`, Mar 1–20 | where the page ranked before the decision moment |
| **Feature (4)** | `days_f20` = days with ≥1 impression, Mar 1–20 | presence/consistency before the decision moment |
| **Feature (5)** | `past_momentum` = rate(Mar 11–20) ÷ rate(Mar 1–10) | trend *inside* the feature window — both halves end before the decision moment |
| **Label** | `decline_next11` = rate(Mar 21–31) < 0.8 × rate(Mar 11–20) | the observed future outcome; computed only from post-decision days (+ its own baseline) |
| **Excluded** | all `ga4_*` / `sessions_*` columns | three-valued availability (`TRUE`/`FALSE`/`NULL`) needs its own contract; engagement side deferred to the capstone |
| **Excluded** | `fact_content_query_90d` columns | fixed 90-day window overlaps the label period → leakage risk (see §1.5) |
| **Excluded** | anything from `month=2026-06` / `_sample` | sealed test month |

The label's source quantities (`imp_late`, `imp_mid`) are **never** features — the guard below
enforces it in code, not just in prose.


In [4]:
FEATURES = ["log_imp_f20", "ctr_f20", "pos_f20", "days_f20", "past_momentum"]
LABEL_SOURCES = ["imp_late", "imp_mid"]   # the label is computed from these two windows

assert not (set(FEATURES) & set(LABEL_SOURCES)), "label source leaked into features"
assert len(FEATURES) == 5, "the card says five features, max"
print("feature list:", FEATURES)
print("leakage guard holds:", not (set(FEATURES) & set(LABEL_SOURCES)))

feature list: ['log_imp_f20', 'ctr_f20', 'pos_f20', 'days_f20', 'past_momentum']
leakage guard holds: True


## 3. Verify it with queries (grain, counts, availability)

Three claims, three queries, run on the same `month=2026-03` partition.


**Query 1 — the grain holds.** Claim: one row really is one page-day. If true,
grouping by (`report_date`, `client_hash_id`, `content_hash_id`) must return **zero**
duplicate groups.


In [5]:
dupes = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"duplicate page-day groups returned: {len(dupes)} (0 means the grain holds)")
dupes

duplicate page-day groups returned: 0 (0 means the grain holds)


,client_hash_id,content_hash_id,report_date,c


**Query 2 — my slice's size and date span.** Claim: the partition covers exactly March 2026,
~9.8M page-days, and how many pages/clients that collapses to.


In [6]:
span = con.sql(f"""
    SELECT COUNT(*)                                         AS rows_page_days,
           MIN(report_date)                                 AS first_day,
           MAX(report_date)                                 AS last_day,
           COUNT(DISTINCT client_hash_id)                   AS clients,
           COUNT(DISTINCT content_hash_id)                  AS pages
    FROM {FACT_MAR}
""").df()
span

,rows_page_days,first_day,last_day,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3 — availability, checked with IS TRUE.** Claim: neither analytics nor search fields are
reliably present — both flags are three-valued (`TRUE` / `FALSE` / `NULL`), so only an `IS TRUE`
filter tells me how many rows genuinely carry data. A `= FALSE` filter would silently drop the
NULL rows and miscount. This measured scarcity is why my five features are search-side only.


In [7]:
avail = con.sql(f"""
    SELECT COUNT(*)                                                    AS rows_total,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)          AS ga4_is_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE)         AS ga4_is_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)          AS ga4_is_null,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)          AS gsc_is_true
    FROM {FACT_MAR}
""").df()
avail["ga4_true_share"] = avail["ga4_is_true"] / avail["rows_total"]
print(avail.to_string(index=False))
print("\nsurviving an IS TRUE filter:", f"{int(avail['ga4_is_true'].iloc[0]):,} of "
      f"{int(avail['rows_total'].iloc[0]):,} rows ({avail['ga4_true_share'].iloc[0]:.1%})")

 rows_total  ga4_is_true  ga4_is_false  ga4_is_null  gsc_is_true  ga4_true_share
    9841378       413966       6408671      3018741      3611061        0.042064

surviving an IS TRUE filter: 413,966 of 9,841,378 rows (4.2%)


### 3b. Five features (max) — the lane's frame for month=2026-03

One SQL pass aggregates page-days → pages; pandas derives the five features and the label.
Pool filter: pages with ≥ 100 impressions in the baseline window (Mar 11–20) — below that, a
"decline" is noise, and those pages aren't realistic refresh candidates anyway. This filter also
does the availability work implicitly: rows with `gsc_data_available` not TRUE are zero-filled
(Query 3), so only genuinely tracked pages can clear a 100-impression bar.

**Every feature is knowable at the decision moment (end of Mar 20):**

- `log_imp_f20` — knowable at the decision moment because it sums impressions over Mar 1–20, which ends before it.
- `ctr_f20` — knowable at the decision moment because clicks and impressions both accrue by Mar 20.
- `pos_f20` — knowable at the decision moment because it averages positions already recorded by Mar 20 (missing → filled 100 = "no measurable ranking").
- `days_f20` — knowable at the decision moment because it counts days already elapsed, out of 20.
- `past_momentum` — knowable at the decision moment because it compares Mar 11–20 vs Mar 1–10; the newer half still ends at Mar 20. This is the *honest cousin* of a trend feature — the trap in §3c uses the future window instead.


In [8]:
frame = con.sql(f"""
    SELECT client_hash_id,
           content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_impressions ELSE 0 END) AS imp_f20,
           SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_clicks     ELSE 0 END) AS clk_f20,
           AVG(CASE WHEN report_date <= DATE '2026-03-20' AND gsc_avg_position > 0
                    THEN gsc_avg_position END)                                            AS pos_raw,
           COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-20' AND gsc_impressions > 0
                               THEN report_date END)                                       AS days_f20,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10'
                    THEN gsc_impressions ELSE 0 END)                                       AS imp_d01_10,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-11' AND DATE '2026-03-20'
                    THEN gsc_impressions ELSE 0 END)                                       AS imp_mid,
           SUM(CASE WHEN report_date >  DATE '2026-03-20'
                    THEN gsc_impressions ELSE 0 END)                                       AS imp_late
    FROM {FACT_MAR}
    GROUP BY 1, 2
""").df()

pool = frame[frame["imp_mid"] >= 100].copy()

import numpy as np

pool["log_imp_f20"]  = np.log1p(pool["imp_f20"])
pool["ctr_f20"]      = pool["clk_f20"] / pool["imp_f20"].replace(0, np.nan)
pool["ctr_f20"]      = pool["ctr_f20"].fillna(0.0)
pool["pos_f20"]      = pool["pos_raw"].fillna(100.0)
pool["past_momentum"] = (pool["imp_mid"] / 10.0) / (pool["imp_d01_10"] / 10.0 + 1e-9)

rate_late = pool["imp_late"] / 11.0
rate_mid  = pool["imp_mid"] / 10.0
pool["decline_next11"] = (rate_late < 0.8 * rate_mid).astype(int)

print(f"page-days aggregated into one row per page: {len(frame):,}")
print(f"pool (imp_mid >= 100): {len(pool):,} pages | decline_next11 rate: {pool['decline_next11'].mean():.1%}")
pool[["client_hash_id", "content_hash_id"] + FEATURES + ["decline_next11"]].head()

page-days aggregated into one row per page: 331,437
pool (imp_mid >= 100): 71,569 pages | decline_next11 rate: 26.0%


,client_hash_id,content_hash_id,log_imp_f20,ctr_f20,pos_f20,days_f20,past_momentum,decline_next11
0,client_62f4a7e64f5e0096,content_1a0cb7648dc42bd1,8.045268,0.000962,2.588844,20,0.898904,0
1,client_62f4a7e64f5e0096,content_2bfa1c2bce65b610,5.627621,0.003610,6.847741,20,1.689320,1
4,client_62f4a7e64f5e0096,content_5cb7083595d4078e,6.690842,0.000000,8.645604,20,0.856813,1
5,client_62f4a7e64f5e0096,content_9c974c0f40eafbb6,7.376508,0.003131,2.001330,20,1.579968,1
6,client_62f4a7e64f5e0096,content_6915e1aa12cfd969,7.596894,0.000502,2.658928,20,0.605645,0


### 3c. The trap — one deliberate, label-derived column

The experiment: train a quick model honestly, then add **one** column computed from the label's
own inputs — `trend_ratio_leak = rate(Mar 21–31) ÷ rate(Mar 11–20)` — and watch the score jump
toward perfect. Then delete it and keep the honest number. Same recipe as notebook 02's lesson,
performed on real warehouse data.

Split: **client-grouped** holdout (~25% of clients), so no client appears on both sides.
Metrics: Precision@50 (my lane's number — the reviewer's top-50 queue) and ROC AUC.


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def quick_score(df, cols):
    X, y, groups = df[cols], df["decline_next11"], df["client_hash_id"]
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups))
    model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    model.fit(X.iloc[tr], y.iloc[tr])
    prob = model.predict_proba(X.iloc[te])[:, 1]
    return precision_at_k(y.iloc[te].to_numpy(), prob), roc_auc_score(y.iloc[te], prob)

p50, auc = quick_score(pool, FEATURES)
HONEST = (p50, auc)
print(f"HONEST  (5 features):        Precision@50 = {p50:.2f} | ROC AUC = {auc:.3f}")

HONEST  (5 features):        Precision@50 = 0.50 | ROC AUC = 0.609


In [10]:
# The trap: add ONE column computed from the label's own two windows.
pool["trend_ratio_leak"] = (pool["imp_late"] / 11.0) / (pool["imp_mid"] / 10.0)

p50_leak, auc_leak = quick_score(pool, FEATURES + ["trend_ratio_leak"])
print(f"LEAKED (5 + trend_ratio_leak): Precision@50 = {p50_leak:.2f} | ROC AUC = {auc_leak:.3f}")
print("-> the score jumped toward perfect because the column IS the label, re-derived.")

LEAKED (5 + trend_ratio_leak): Precision@50 = 1.00 | ROC AUC = 1.000
-> the score jumped toward perfect because the column IS the label, re-derived.


In [11]:
# Delete the trap, keep the honest number.
pool = pool.drop(columns=["trend_ratio_leak"])
assert "trend_ratio_leak" not in pool.columns

p50_h, auc_h = quick_score(pool, FEATURES)
assert abs(p50_h - HONEST[0]) < 1e-9 and abs(auc_h - HONEST[1]) < 1e-9, "honest score did not reproduce"
base_rate = pool["decline_next11"].mean()
print(f"RESTORED after delete:       Precision@50 = {p50_h:.2f} | ROC AUC = {auc_h:.3f}")
print(f"\nTHE NUMBER I KEEP: Precision@50 = {p50_h:.2f}, ROC AUC = {auc_h:.3f} "
      f"(leaked run scored {p50_leak:.2f}/{auc_leak:.3f} - discarded)")
print(f"context: random picks would score the {base_rate:.0%} base rate at the top of the queue; "
      f"{p50_h:.2f} is directional decision-support, not a finished model.")

RESTORED after delete:       Precision@50 = 0.50 | ROC AUC = 0.609

THE NUMBER I KEEP: Precision@50 = 0.50, ROC AUC = 0.609 (leaked run scored 1.00/1.000 - discarded)
context: random picks would score the 26% base rate at the top of the queue; 0.50 is directional decision-support, not a finished model.


## 4. Data limits

**Named limitation of this slice: engagement coverage is partial and patterned.** Query 3
measured it: only ~4% of March page-days carry `ga4_data_available IS TRUE`, and availability
follows each client's onboarding date, not chance. So my five-feature frame is deliberately
search-side only — engagement-side refresh signals (sessions, scroll depth, AI referrals) cannot
be built for the whole frame without their own availability contract. Deferred to the capstone,
with `IS TRUE` filters from day one.

Two more limits I keep visible:

- **Unbalanced panel:** clients enter the warehouse when their tracking starts, so equal calendar
  windows contain unequal histories (measured below against `dim_clients`).
- **Sealed final month:** June 2026 (`_sample`) is the natural outcome window of any
  past→future label — used for mechanics practice only, never for developing label logic.


In [12]:
total_clients = con.sql(f"SELECT COUNT(*) FROM {DIM_CLIENTS}").fetchone()[0]
in_march = pool["client_hash_id"].nunique()
print(f"clients in dim_clients: {total_clients} | clients with pages in the March pool: {in_march} "
      f"({in_march/total_clients:.0%})")

clients in dim_clients: 104 | clients with pages in the March pool: 37 (36%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers (§1) — row meaning, tables, window, label/proxy, exclusion
- [x] Exactly three verification queries with outputs visible (§3) — grain, count+span, availability via `IS TRUE`
- [x] Five-feature frame with an "available when?" line per feature (§3b)
- [x] Deliberate-leak experiment shown, then removed; honest number kept (§3c)
- [x] One named limitation (§4)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, tokens, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed under `work/notebooks/` — then submit the repo URL on the card. Done.
